# EDA — NO2.csv
Score de Vivabilité · Concentrations NO₂ sur les tronçons du Périphérique parisien

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

## 1. Chargement

In [ ]:
FILE = 'architecture-data/brute/score_de_vivabilité/NO2.csv'

df = pd.read_csv(FILE, sep=None, engine='python')
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)
df['time'] = pd.to_datetime(df['time'], errors='coerce')

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

## 2. Infos générales

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Valeurs manquantes — critique pour ce dataset

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, max(3, len(df.columns) * 0.45)))
colors = ['#D85A30' if v > 50 else '#BA7517' if v > 20 else '#1D9E75' for v in missing.values]
ax.barh(missing.index, missing.values, color=colors)
ax.set_xlabel('% manquant')
ax.set_title('Valeurs manquantes par colonne (rouge > 50%, orange > 20%)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.axvline(50, color='#D85A30', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

display(missing.rename('% manquant').to_frame())

## 4. Doublons

In [ ]:
n_dup = df.duplicated().sum()
print(f'Doublons : {n_dup} ({n_dup/len(df)*100:.2f}%)')
if n_dup > 0:
    display(df[df.duplicated(keep=False)].head(10))

## 5. Couverture temporelle

In [ ]:
print(f'Première mesure : {df["time"].min()}')
print(f'Dernière mesure : {df["time"].max()}')
print(f'Durée couverte  : {(df["time"].max() - df["time"].min()).days} jours')
print(f'Nb mesures      : {len(df):,}')
freq = df['time'].diff().dt.total_seconds().dropna().mode()
if not freq.empty:
    print(f'Fréquence mode  : {freq.values[0]/3600:.1f}h')

## 6. Distribution NO₂ par tronçon

In [ ]:
troncons = [c for c in df.columns if c != 'time']

fig, ax = plt.subplots(figsize=(10, 5))
df[troncons].boxplot(ax=ax)
ax.axhline(40, color='red', linestyle='--', linewidth=1.2, label='Seuil OMS annuel 40 µg/m³')
ax.axhline(200, color='orange', linestyle='--', linewidth=1, label='Seuil horaire 200 µg/m³')
ax.set_title('Distribution NO₂ par tronçon du Périphérique (µg/m³)')
ax.set_ylabel('NO₂ (µg/m³)')
ax.legend(fontsize=8)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

stats = df[troncons].describe().T
stats['> seuil OMS (%)'] = (df[troncons] > 40).mean() * 100
display(stats.round(2))

## 7. Évolution temporelle (moyenne mobile 24h)

In [ ]:
df_sorted = df.sort_values('time').set_index('time')
colors = ['#378ADD','#1D9E75','#D85A30','#BA7517','#888780','#534AB7','#D4537E','#2E8B57']

fig, ax = plt.subplots(figsize=(14, 5))
for col, color in zip(troncons, colors):
    rolling = df_sorted[col].rolling('24h', min_periods=6).mean()
    ax.plot(rolling.index, rolling, label=col, color=color, linewidth=1, alpha=0.8)
ax.axhline(40, color='red', linestyle='--', linewidth=1, label='Seuil OMS 40 µg/m³')
ax.set_title('NO₂ (moyenne mobile 24h) par tronçon')
ax.set_ylabel('NO₂ µg/m³')
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
plt.show()

## 8. Profil horaire moyen

In [ ]:
df_h = df_sorted.copy()
df_h['heure'] = df_h.index.hour
hourly = df_h.groupby('heure')[troncons].mean()

fig, ax = plt.subplots(figsize=(10, 5))
for col, color in zip(troncons, colors):
    ax.plot(hourly.index, hourly[col], label=col, color=color, marker='o', markersize=3)
ax.set_title('Profil horaire moyen NO₂ par tronçon')
ax.set_xlabel('Heure de la journée')
ax.set_ylabel('NO₂ moyen (µg/m³)')
ax.set_xticks(range(0, 24))
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 9. Corrélation entre tronçons

In [ ]:
corr = df[troncons].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax,
            vmin=0.5, vmax=1, linewidths=0.5)
ax.set_title('Corrélation NO₂ entre tronçons')
plt.tight_layout()
plt.show()

## 10. Résumé + plan Silver

In [ ]:
pct_oms = (df[troncons] > 40).mean().mean() * 100
print('=' * 55)
print('RÉSUMÉ EDA — NO2.csv')
print('=' * 55)
print(f'  Lignes (mesures)     : {len(df):,}')
print(f'  Tronçons             : {len(troncons)}')
print(f'  Couverture           : {df["time"].min()} → {df["time"].max()}')
print(f'  NaN par tronçon      : ~1.1% (très faible)')
print(f'  % mesures > OMS 40   : {pct_oms:.1f}% en moyenne')
print('=' * 55)
print()
print('ACTIONS SILVER REQUISES :')
print('  → Interpoler les ~1.1% NaN (interpolation linéaire temporelle)')
print('  → Calculer moyenne journalière et mensuelle par tronçon')
print('  → Identifier les tronçons chroniquement > seuil OMS')
print('  → Agréger en score NO₂ = moyenne annuelle / 40 µg/m³')
print('  → Export Parquet pour Gold scoring vivabilité')